# Subreddit review: v4 sample, activity before first accusation

One row per subreddit the v4 sample (`sample_1000_v4_output/`) posted or commented in **before each account's first
accusation** (`created_utc < first_mention_adj_utc`), for drafting subreddit categories (general, karma farming,
political, adult, crypto, sales, blog/YouTube promotion, ...).

Counts are of sample accounts, posts and comments; example titles are up to three distinct post titles. Written to
`subreddit_review_output/` as a spreadsheet for review; `save_new()` refuses to overwrite.

In [ ]:
# Libraries and folders.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
os.makedirs(OUT_DIR, exist_ok=True)

# Write an Excel file only if it does not exist yet (never overwrite a review sheet).
def save_new_xlsx(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_excel(path, index=False, freeze_panes=(1, 1))
    print(f'wrote {path}  {df.shape}')

In [ ]:
# The v4 sample's accounts, and their posts and comments.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
posts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_posts.parquet'),
                        columns=['outer_key', 'created_utc', 'subreddit', 'title', 'over_18', 'is_self', 'domain'])
comments = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_comments.parquet'), columns=['outer_key', 'created_utc', 'subreddit'])
# Keep only activity before each account's edit-adjusted first accusation.
posts = posts[posts['created_utc'] < posts['outer_key'].map(accounts['first_mention_adj_utc'])]
comments = comments[comments['created_utc'] < comments['outer_key'].map(accounts['first_mention_adj_utc'])]
print(f'pre-accusation posts {len(posts):,}, comments {len(comments):,}')

In [ ]:
# One row per subreddit: how many accounts used it, post and comment counts, and share of NSFW posts.
posts['nsfw'] = posts['over_18'].astype(str).eq('True')
links = posts[~posts['is_self'].astype(str).eq('True')]
p = posts.groupby('subreddit').agg(posts=('subreddit', 'size'), accounts_posting=('outer_key', 'nunique'), nsfw_post_share=('nsfw', 'mean'))
c = comments.groupby('subreddit').agg(comments=('subreddit', 'size'), accounts_commenting=('outer_key', 'nunique'))
both = pd.concat([posts[['subreddit', 'outer_key']], comments[['subreddit', 'outer_key']]])
review = (pd.DataFrame({'accounts_total': both.groupby('subreddit')['outer_key'].nunique()})
            .join(p).join(c).fillna({'posts': 0, 'comments': 0, 'accounts_posting': 0, 'accounts_commenting': 0}))
# For hand review: the three most common link domains and three example post titles (cut to 120 characters).
review['top_link_domains'] = links.groupby('subreddit')['domain'].agg(lambda d: ', '.join(d.value_counts().head(3).index))
review['example_titles'] = posts.groupby('subreddit')['title'].agg(
    lambda t: ' | '.join(t.dropna().drop_duplicates().head(3).str.slice(0, 120)))
for col in ['posts', 'comments', 'accounts_posting', 'accounts_commenting']:
    review[col] = review[col].astype(int)
# Most-used subreddits first.
review = (review.reset_index().rename(columns={'index': 'subreddit'})
                .sort_values(['accounts_total', 'posts', 'comments'], ascending=False).reset_index(drop=True))
review.insert(1, 'category', '')                       # left blank for review
print(f'{len(review):,} subreddits; used by 1 account only: {(review.accounts_total == 1).sum():,}')
review.head(40)

In [ ]:
# Save the review sheet.
save_new_xlsx(review, 'subreddits_v4_pre_accusation.xlsx')